## Scenario 1: A single data scientist participating in an ML competition

MLflow setup:
* Tracking server: no
* Backend store: local filesystem
* Artifacts store: local filesystem

The experiments can be explored locally by launching the MLflow UI.

In [2]:
import mlflow

In [3]:
print(f"tracking URI: '{mlflow.get_tracking_uri()}'")

tracking URI: 'sqlite:////home/leandro/estudos/mlops-zoomcamp/02-experiment-tracking/running-mlflow-examples/mlflow.db'


In [3]:
mlflow.search_experiments()

[<Experiment: artifact_location='/home/leandro/estudos/mlops-zoomcamp/02-experiment-tracking/running-mlflow-examples/mlruns/1', creation_time=1793019809853, effective_trace_archival_retention=None, experiment_id='1', last_update_time=1793019809853, lifecycle_stage='active', name='my-experiment-1', tags={}, trace_location=None, workspace='default'>,
 <Experiment: artifact_location='/home/leandro/estudos/mlops-zoomcamp/02-experiment-tracking/running-mlflow-examples/mlruns/0', creation_time=1793019273411, effective_trace_archival_retention=None, experiment_id='0', last_update_time=1793019273411, lifecycle_stage='active', name='Default', tags={}, trace_location=None, workspace='default'>]

### Creating an experiment and logging a new run

In [4]:
from sklearn.linear_model import LogisticRegression
from sklearn.datasets import load_iris
from sklearn.metrics import accuracy_score

mlflow.set_experiment("my-experiment-1")

with mlflow.start_run() as run:

    X, y = load_iris(return_X_y=True)

    params = {"C": 0.1, "random_state": 42}
    mlflow.log_params(params)

    lr = LogisticRegression(**params).fit(X, y)
    y_pred = lr.predict(X)
    mlflow.log_metric("accuracy", accuracy_score(y, y_pred))

    mlflow.sklearn.log_model(lr, name="models")
    print(f"default artifacts URI: '{mlflow.get_artifact_uri()}'")

2026/10/26 11:21:01 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.


default artifacts URI: '/home/leandro/estudos/mlops-zoomcamp/02-experiment-tracking/running-mlflow-examples/mlruns/1/3dabf96c5eee42e987a66435c5ae98b4/artifacts'


In [6]:
# run_id = run.info.run_id
# print(run_id)
# print(mlflow.get_run(run_id).data.params)

experiments = mlflow.search_experiments()

# for experiment in experiments:
#     print(experiment.experiment_id, experiment.name)

experiment_id = mlflow.search_experiments(filter_string="name LIKE 'my-experiment-1'")[0].experiment_id
run_id = mlflow.search_runs(experiment_id)

print(run_id)


# print(run)
# print(experiment.data.metrics)
# print(experiment.data.tags)

                             run_id experiment_id    status  \
0  3dabf96c5eee42e987a66435c5ae98b4             1  FINISHED   
1  0a98532b551a4ac3b11ab040f7ef99c3             1    KILLED   
2  b75b9edd7685443387c1c5797470add9             1  FINISHED   

                                        artifact_uri  \
0  /home/leandro/estudos/mlops-zoomcamp/02-experi...   
1  /home/leandro/estudos/mlops-zoomcamp/02-experi...   
2  /home/leandro/estudos/mlops-zoomcamp/02-experi...   

                        start_time                         end_time  \
0 2026-10-26 14:20:48.955000+00:00 2026-10-26 14:21:01.510000+00:00   
1 2026-10-26 13:28:50.705000+00:00 2026-10-26 17:13:43.548000+00:00   
2 2026-10-26 13:03:30.300000+00:00 2026-10-26 13:03:46.773000+00:00   

   metrics.accuracy params.C params.random_state tags.mlflow.runName  \
0              0.96      0.1                  42   adaptable-hen-345   
1               NaN      NaN                 NaN   honorable-ant-216   
2              0.96  

In [5]:
client = mlflow.tracking.MlflowClient()
client.set_terminated(run_id="0a98532b551a4ac3b11ab040f7ef99c3", status="KILLED")

### Interacting with the model registry

In [7]:
from mlflow.tracking import MlflowClient


client = MlflowClient()

In [8]:
from mlflow.exceptions import MlflowException

try:
    client.search_registered_models()
except MlflowException:
    print("It's not possible to access the model registry :(")

To check this experiment at the `mlflow ui`, you need:
1. to go to this folder path at `cd 02-experiment-tracking/running-mlflow-examples/`; and 
2. run `mlflow ui`.